In [1]:
from datasets import load_dataset

corpus = load_dataset(
    "BeIR/scidocs",
    "corpus"
)

queries = load_dataset(
    "BeIR/scidocs",
    "queries"
)

import pandas as pd
import numpy as np

corpus_df = corpus["corpus"].to_pandas()
queries_df = queries["queries"].to_pandas()

corpus_df["text_words"] = corpus_df["text"].str.split().str.len()
corpus_df["title_words"] = corpus_df["title"].str.split().str.len()

queries_df["text_words"] = queries_df["text"].str.split().str.len()
queries_df["title_words"] = queries_df["title"].str.split().str.len()

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.7MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating corpus split:   0%|          | 0/25657 [00:00<?, ? examples/s]

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 92.6kB            

queries/queries-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating queries split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Max sequence length:", model.max_seq_length)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Max sequence length: 256
Embedding dimension: 384


/tmp/ipykernel_2272/3460295810.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [ ]:
corpus_embeddings = model.encode(
    corpus_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print(corpus_embeddings.shape)

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

torch.Size([25657, 384])


In [ ]:
query = queries_df.iloc[0]["text"]

query_embedding = model.encode(
    query,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Query:", query)
print("Embedding shape:", query_embedding.shape)

Query: A Direct Search Method to solve Economic Dispatch Problem with Valve-Point Effect
Embedding shape: torch.Size([384])


In [ ]:
from sentence_transformers import util

hits = util.semantic_search(
    query_embedding,
    corpus_embeddings,
    top_k=10
)[0]

In [ ]:
hits

[{'corpus_id': 1, 'score': 0.5885633826255798},
 {'corpus_id': 23875, 'score': 0.42465728521347046},
 {'corpus_id': 14045, 'score': 0.4246130585670471},
 {'corpus_id': 20570, 'score': 0.4216872453689575},
 {'corpus_id': 14060, 'score': 0.42166614532470703},
 {'corpus_id': 19958, 'score': 0.41366708278656006},
 {'corpus_id': 10372, 'score': 0.40498608350753784},
 {'corpus_id': 22109, 'score': 0.40021950006484985},
 {'corpus_id': 11732, 'score': 0.4002060890197754},
 {'corpus_id': 12830, 'score': 0.3988041877746582}]

In [ ]:
for rank, hit in enumerate(hits, start=1):
    doc = corpus_df.iloc[hit["corpus_id"]]

    print(f"\nRank {rank}")
    print(f"Score: {hit['score']:.4f}")
    print(f"Corpus ID: {doc['_id']}")
    print(f"Title: {doc['title']}")
    print(f"Text: {doc['text'][:300]}...")


Rank 1
Score: 0.5886
Corpus ID: 86e87db2dab958f1bd5877dc7d5b8105d6e31e46
Title: A Hybrid EP and SQP for Dynamic Economic Dispatch with Nonsmooth Fuel Cost Function
Text: Dynamic economic dispatch (DED) is one of the main functions of power generation operation and control. It determines the optimal settings of generator units with predicted load demand over a certain period of time. The objective is to operate an electric power system most economically while the sys...

Rank 2
Score: 0.4247
Corpus ID: 9cf8464e03a2c78e2816b0478bd323bf40ab802b
Title: A simple and effective iterated greedy algorithm for the permutation flowshop scheduling problem
Text: Over the last decade many metaheuristics have been applied t o the flowshop scheduling problem, ranging from Simulated Annealing or Tabu Search to complex hybrid techniques. Some of these methods provide excellent effectiveness and e fficiency at the expense of being utterly complicated. In fact, se...

Rank 3
Score: 0.4246
Corpus ID: 91de

In [ ]:
query_embeddings = model.encode(
    queries_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
all_hits = util.semantic_search(
    query_embeddings,
    corpus_embeddings,
    top_k=10
)

In [2]:
!pip install -q beir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 28.1 MB/s eta 0:00:00


In [3]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

dataset = "scidocs"

url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

data_path = util.download_and_unzip(
    url,
    "datasets"
)

corpus_beir, queries_beir, qrels = GenericDataLoader(
    data_folder=data_path
).load(split="test")

datasets/scidocs.zip:   0%|          | 0.00/136M [00:00<?, ?iB/s]

  0%|          | 0/25657 [00:00<?, ?it/s]

In [4]:
import pandas as pd

qrels_records = []
for query_id, doc_scores in qrels.items():
    for doc_id, score in doc_scores.items():
        qrels_records.append({'query_id': query_id, 'doc_id': doc_id, 'score': score})

qrels_df = pd.DataFrame(qrels_records)



In [5]:
qrels_dict = (
    qrels_df.groupby("query_id")["doc_id"]
    .apply(set)
    .to_dict()
)

In [6]:
corpus_id_to_doc_id = dict(
    enumerate(corpus_df["_id"])
)



In [ ]:
retrieval_labels = []

for query_idx, hits in enumerate(all_hits):

    query_id = queries_df.iloc[query_idx]["_id"]
    relevant_docs = qrels_dict[query_id]

    labels = []

    for hit in hits:
        doc_id = corpus_id_to_doc_id[hit["corpus_id"]]

        if doc_id in relevant_docs:
            labels.append(1)
        else:
            labels.append(0)

    retrieval_labels.append(labels)

In [ ]:
retrieval_labels[0]

[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [ ]:
recall_scores = []

for query_idx, labels in enumerate(retrieval_labels):

    query_id = queries_df.iloc[query_idx]["_id"]

    total_relevant = len(qrels_dict[query_id])
    retrieved_relevant = sum(labels)

    recall = retrieved_relevant / total_relevant

    recall_scores.append(recall)

mean_recall_at_10 = sum(recall_scores) / len(recall_scores)

print(f"Recall@10: {mean_recall_at_10:.4f}")

Recall@10: 0.2207


In [ ]:
retrieval_labels = []

for query_idx, hits in enumerate(all_hits):

    query_id = queries_df.iloc[query_idx]["_id"]
    relevant_docs = qrels_dict[query_id]

    labels = []

    for hit in hits:
        doc_id = corpus_id_to_doc_id[hit["corpus_id"]]

        if doc_id in relevant_docs:
            labels.append(1)
        else:
            labels.append(0)

    retrieval_labels.append(labels)

In [ ]:
mrr_scores = []

for labels in retrieval_labels:

    reciprocal_rank = 0

    for rank, label in enumerate(labels, start=1):

        if label == 1:
            reciprocal_rank = 1 / rank
            break

    mrr_scores.append(reciprocal_rank)

mean_mrr_at_10 = sum(mrr_scores) / len(mrr_scores)

print(f"MRR@10: {mean_mrr_at_10:.4f}")

MRR@10: 0.3431


In [ ]:
import math

ndcg_scores = []

for labels in retrieval_labels:

    dcg = 0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            dcg += 1 / math.log2(rank + 1)

    num_relevant_retrieved = sum(labels)

    idcg = 0

    for rank in range(1, num_relevant_retrieved + 1):
        idcg += 1 / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0

    ndcg_scores.append(ndcg)

mean_ndcg_at_10 = sum(ndcg_scores) / len(ndcg_scores)

print(f"nDCG@10: {mean_ndcg_at_10:.4f}")

nDCG@10: 0.3970


In [ ]:
# Map IDs to actual text

query_text_map = dict(
    zip(queries_df["_id"], queries_df["text"])
)

doc_text_map = dict(
    zip(corpus_df["_id"], corpus_df["text"])
)

In [ ]:
query_id = qrels_df.iloc[0]["query_id"]
doc_id = qrels_df.iloc[0]["doc_id"]

print("Query:")
print(query_text_map[query_id])

print("\nDocument:")
print(doc_text_map[doc_id])

Query:
A Direct Search Method to solve Economic Dispatch Problem with Valve-Point Effect

Document:
An evolutionary recurrent network which automates the design of recurrent neural/fuzzy networks using a new evolutionary learning algorithm is proposed in this paper. This new evolutionary learning algorithm is based on a hybrid of genetic algorithm (GA) and particle swarm optimization (PSO), and is thus called HGAPSO. In HGAPSO, individuals in a new generation are created, not only by crossover and mutation operation as in GA, but also by PSO. The concept of elite strategy is adopted in HGAPSO, where the upper-half of the best-performing individuals in a population are regarded as elites. However, instead of being reproduced directly to the next generation, these elites are first enhanced. The group constituted by the elites is regarded as a swarm, and each elite corresponds to a particle within it. In this regard, the elites are enhanced by PSO, an operation which mimics the maturing p

In [ ]:
query_text = "A Direct Search Method to solve Economic Dispatch Problem with Valve-Point Effect"

query_row = queries_df[
    queries_df["text"] == query_text
]

print(query_row[["_id", "text"]])

                                        _id  \
0  78495383450e02c5fe817e408726134b3084905d   

                                                text  
0  A Direct Search Method to solve Economic Dispa...  


In [ ]:
query_id = query_row.iloc[0]["_id"]

print("Query ID:", query_id)

print(
    qrels_df[qrels_df["query_id"] == query_id].head(10)
)

Query ID: 78495383450e02c5fe817e408726134b3084905d
                                   query_id  \
0  78495383450e02c5fe817e408726134b3084905d   
1  78495383450e02c5fe817e408726134b3084905d   
2  78495383450e02c5fe817e408726134b3084905d   
3  78495383450e02c5fe817e408726134b3084905d   
4  78495383450e02c5fe817e408726134b3084905d   
5  78495383450e02c5fe817e408726134b3084905d   
6  78495383450e02c5fe817e408726134b3084905d   
7  78495383450e02c5fe817e408726134b3084905d   
8  78495383450e02c5fe817e408726134b3084905d   
9  78495383450e02c5fe817e408726134b3084905d   

                                     doc_id  score  
0  632589828c8b9fca2c3a59e97451fde8fa7d188d      1  
1  86e87db2dab958f1bd5877dc7d5b8105d6e31e46      1  
2  2a047d8c4c2a4825e0f0305294e7da14f8de6fd3      1  
3  506172b0e0dd4269bdcfe96dda9ea9d8602bbfb6      1  
4  51317b6082322a96b4570818b7a5ec8b2e330f2f      1  
5  857a8c6c46b0a85ed6019f5830294872f2f1dcf5      0  
6  12f107016fd3d062dff88a00d6b0f5f81f00522d      0  
7  1ae0

In [ ]:
positive_qrels = qrels_df[qrels_df["score"] == 1]

qrels_dict = (
    positive_qrels.groupby("query_id")["doc_id"]
    .apply(set)
    .to_dict()
)

print("Total positive pairs:", len(positive_qrels))
print("Queries:", len(qrels_dict))

Total positive pairs: 4928
Queries: 1000


In [ ]:
query_id = "78495383450e02c5fe817e408726134b3084905d"

print(
    qrels_df[qrels_df["query_id"] == query_id]
)

                                    query_id  \
0   78495383450e02c5fe817e408726134b3084905d   
1   78495383450e02c5fe817e408726134b3084905d   
2   78495383450e02c5fe817e408726134b3084905d   
3   78495383450e02c5fe817e408726134b3084905d   
4   78495383450e02c5fe817e408726134b3084905d   
5   78495383450e02c5fe817e408726134b3084905d   
6   78495383450e02c5fe817e408726134b3084905d   
7   78495383450e02c5fe817e408726134b3084905d   
8   78495383450e02c5fe817e408726134b3084905d   
9   78495383450e02c5fe817e408726134b3084905d   
10  78495383450e02c5fe817e408726134b3084905d   
11  78495383450e02c5fe817e408726134b3084905d   
12  78495383450e02c5fe817e408726134b3084905d   
13  78495383450e02c5fe817e408726134b3084905d   
14  78495383450e02c5fe817e408726134b3084905d   
15  78495383450e02c5fe817e408726134b3084905d   
16  78495383450e02c5fe817e408726134b3084905d   
17  78495383450e02c5fe817e408726134b3084905d   
18  78495383450e02c5fe817e408726134b3084905d   
19  78495383450e02c5fe817e408726134b3084

In [ ]:
from sentence_transformers import util
hard_negative_hits = util.semantic_search(
    query_embeddings,
    corpus_embeddings,
    top_k=50
)

In [ ]:
hard_negatives = []

for query_idx, hits in enumerate(hard_negative_hits):

    query_id = queries_df.iloc[query_idx]["_id"]
    relevant_docs = qrels_dict[query_id]

    for hit in hits:

        doc_id = corpus_id_to_doc_id[hit["corpus_id"]]

        if doc_id not in relevant_docs:
            hard_negatives.append({
                "query_id": query_id,
                "doc_id": doc_id,
                "score": hit["score"]
            })

hard_negatives_df = pd.DataFrame(hard_negatives)

print(hard_negatives_df.shape)
print(hard_negatives_df.head())

(48026, 3)
                                   query_id  \
0  78495383450e02c5fe817e408726134b3084905d   
1  78495383450e02c5fe817e408726134b3084905d   
2  78495383450e02c5fe817e408726134b3084905d   
3  78495383450e02c5fe817e408726134b3084905d   
4  78495383450e02c5fe817e408726134b3084905d   

                                     doc_id     score  
0  9cf8464e03a2c78e2816b0478bd323bf40ab802b  0.424657  
1  91de962e115bcf65eaf8579471a818ba8c5b0ea6  0.424613  
2  cd31ecb3b58d1ec0d8b6e196bddb71dd6a921b6d  0.421687  
3  3fd46ca896d023df8c8af2b3951730d8c38defdd  0.421666  
4  1f376c10b20319121102db78e7790cf47d8fa046  0.413667  


In [ ]:
from sklearn.model_selection import train_test_split

query_ids = queries_df["_id"].tolist()

train_query_ids, temp_query_ids = train_test_split(
    query_ids,
    test_size=0.2,
    random_state=42
)

val_query_ids, test_query_ids = train_test_split(
    temp_query_ids,
    test_size=0.5,
    random_state=42
)

print("Train queries:", len(train_query_ids))
print("Validation queries:", len(val_query_ids))
print("Test queries:", len(test_query_ids))

Train queries: 800
Validation queries: 100
Test queries: 100


In [ ]:
train_qrels = qrels_df[
    qrels_df["query_id"].isin(train_query_ids)
].copy()

val_qrels = qrels_df[
    qrels_df["query_id"].isin(val_query_ids)
].copy()

test_qrels = qrels_df[
    qrels_df["query_id"].isin(test_query_ids)
].copy()

print("Train qrels:", train_qrels.shape)
print("Validation qrels:", val_qrels.shape)
print("Test qrels:", test_qrels.shape)

Train qrels: (23944, 3)
Validation qrels: (2995, 3)
Test qrels: (2989, 3)


In [ ]:
print("Train score distribution:")
print(train_qrels["score"].value_counts())

print("\nPositive documents per query:")
print(
    train_qrels[train_qrels["score"] == 1]
    .groupby("query_id")
    .size()
    .describe()
)

Train score distribution:
score
0    20000
1     3944
Name: count, dtype: int64

Positive documents per query:
count    800.000000
mean       4.930000
std        0.291902
min        3.000000
25%        5.000000
50%        5.000000
75%        5.000000
max        5.000000
dtype: float64


In [ ]:
train_hard_negatives = hard_negatives_df[
    hard_negatives_df["query_id"].isin(train_query_ids)
].copy()

print("Train hard negatives:", train_hard_negatives.shape)
print(train_hard_negatives.head())

Train hard negatives: (38449, 3)
                                   query_id  \
0  78495383450e02c5fe817e408726134b3084905d   
1  78495383450e02c5fe817e408726134b3084905d   
2  78495383450e02c5fe817e408726134b3084905d   
3  78495383450e02c5fe817e408726134b3084905d   
4  78495383450e02c5fe817e408726134b3084905d   

                                     doc_id     score  
0  9cf8464e03a2c78e2816b0478bd323bf40ab802b  0.424657  
1  91de962e115bcf65eaf8579471a818ba8c5b0ea6  0.424613  
2  cd31ecb3b58d1ec0d8b6e196bddb71dd6a921b6d  0.421687  
3  3fd46ca896d023df8c8af2b3951730d8c38defdd  0.421666  
4  1f376c10b20319121102db78e7790cf47d8fa046  0.413667  


In [ ]:
train_positive = train_qrels[
    train_qrels["score"] == 1
][["query_id", "doc_id"]].copy()

print("Positive pairs:", train_positive.shape)
print(train_positive.head())

Positive pairs: (3944, 2)
                                   query_id  \
0  78495383450e02c5fe817e408726134b3084905d   
1  78495383450e02c5fe817e408726134b3084905d   
2  78495383450e02c5fe817e408726134b3084905d   
3  78495383450e02c5fe817e408726134b3084905d   
4  78495383450e02c5fe817e408726134b3084905d   

                                     doc_id  
0  632589828c8b9fca2c3a59e97451fde8fa7d188d  
1  86e87db2dab958f1bd5877dc7d5b8105d6e31e46  
2  2a047d8c4c2a4825e0f0305294e7da14f8de6fd3  
3  506172b0e0dd4269bdcfe96dda9ea9d8602bbfb6  
4  51317b6082322a96b4570818b7a5ec8b2e330f2f  


In [ ]:
query_lookup = queries_df.set_index("_id")["text"].to_dict()
doc_lookup = corpus_df.set_index("_id")["text"].to_dict()

In [ ]:
train_hard_negatives = train_hard_negatives.sort_values(
    ["query_id", "score"],
    ascending=[True, False]
)

print(train_hard_negatives.head(10))

                                       query_id  \
11448  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
11449  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
11450  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
11451  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
11452  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
11453  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
11454  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
11455  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
11456  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
11457  01273bd34dacfe9ef887b320f36934d2f9fa9b34   

                                         doc_id     score  
11448  a0fd813b9218813e1b020d03a3099de7677dd145  0.407742  
11449  802b80852996d87dc16082b86f6e77115eb6c9a6  0.385802  
11450  0faeda106e40a43a208f76824d8001baac2eefbc  0.370300  
11451  d1b86fa7c4160811de7529bff6bb1cfc206ac1d0  0.368244  
11452  f7686113611ac3d03c1cd412ebf46ba5e35b3071  0.361985  
11453  c6d59c7d23186d4f370d4e5ae4065ffc0efdbbf8  0.352255  
11454  b6f5b165928

In [ ]:
hard_negative_pool = (
    train_hard_negatives
    .groupby("query_id")
    .head(5)
    .reset_index(drop=True)
)

print("Hard negative pool:", hard_negative_pool.shape)
print(hard_negative_pool.head())

Hard negative pool: (4000, 3)
                                   query_id  \
0  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
1  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
2  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
3  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
4  01273bd34dacfe9ef887b320f36934d2f9fa9b34   

                                     doc_id     score  
0  a0fd813b9218813e1b020d03a3099de7677dd145  0.407742  
1  802b80852996d87dc16082b86f6e77115eb6c9a6  0.385802  
2  0faeda106e40a43a208f76824d8001baac2eefbc  0.370300  
3  d1b86fa7c4160811de7529bff6bb1cfc206ac1d0  0.368244  
4  f7686113611ac3d03c1cd412ebf46ba5e35b3071  0.361985  


In [ ]:
import pandas as pd

triplets = []

for query_id, positives in train_positive.groupby("query_id"):

    negatives = hard_negative_pool[
        hard_negative_pool["query_id"] == query_id
    ].copy()

    positives = positives.reset_index(drop=True)
    negatives = negatives.sample(
        n=min(len(positives), len(negatives)),
        random_state=42
    ).reset_index(drop=True)

    n = min(len(positives), len(negatives))

    for i in range(n):
        triplets.append({
            "query_id": query_id,
            "positive_doc_id": positives.iloc[i]["doc_id"],
            "negative_doc_id": negatives.iloc[i]["doc_id"]
        })

train_triplets = pd.DataFrame(triplets)

print("Triplets:", train_triplets.shape)
print(train_triplets.head())

Triplets: (3944, 3)
                                   query_id  \
0  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
1  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
2  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
3  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
4  01273bd34dacfe9ef887b320f36934d2f9fa9b34   

                            positive_doc_id  \
0  2abf2c3e7ebed04e8c09e478157372dda5cb8bc5   
1  2c6c6d3c94322e9ff75ff2143f7028bfab2b3c5f   
2  00a7370518a6174e078df1c22ad366a2188313b5   
3  42d60f7faaa2f6fdd2b928c352d65eb57b4791aa   
4  90614cea8c2ab2bff0343231a26d6d0c9315d6c7   

                            negative_doc_id  
0  802b80852996d87dc16082b86f6e77115eb6c9a6  
1  f7686113611ac3d03c1cd412ebf46ba5e35b3071  
2  0faeda106e40a43a208f76824d8001baac2eefbc  
3  a0fd813b9218813e1b020d03a3099de7677dd145  
4  d1b86fa7c4160811de7529bff6bb1cfc206ac1d0  


In [ ]:
train_triplets["query"] = train_triplets["query_id"].map(
    query_lookup
)

train_triplets["positive"] = train_triplets["positive_doc_id"].map(
    doc_lookup
)

train_triplets["negative"] = train_triplets["negative_doc_id"].map(
    doc_lookup
)

print(train_triplets[
    ["query", "positive", "negative"]
].head(3))

                                         query  \
0  Image-Guided Nanopositioning Scheme for SEM   
1  Image-Guided Nanopositioning Scheme for SEM   
2  Image-Guided Nanopositioning Scheme for SEM   

                                            positive  \
0  Robotics continues to provide researchers with...   
1  In this paper, we have derived analytic expres...   
2  Optical flow cannot be computed locally, since...   

                                            negative  
0  In this article, a methodology to extract Flas...  
1  Multiple problems, including high computationa...  
2  We demonstrate that metal-insulator-metal conf...  


In [ ]:
print("Missing queries:", train_triplets["query"].isna().sum())
print("Missing positives:", train_triplets["positive"].isna().sum())
print("Missing negatives:", train_triplets["negative"].isna().sum())

Missing queries: 0
Missing positives: 0
Missing negatives: 0


In [ ]:
train_pairs = train_positive.copy()

train_pairs["query"] = train_pairs["query_id"].map(
    query_lookup
)

train_pairs["positive"] = train_pairs["doc_id"].map(
    doc_lookup
)

train_pairs = train_pairs[
    ["query", "positive"]
].reset_index(drop=True)

print("Training pairs:", train_pairs.shape)
print(train_pairs.head())

Training pairs: (3944, 2)
                                               query  \
0  A Direct Search Method to solve Economic Dispa...   
1  A Direct Search Method to solve Economic Dispa...   
2  A Direct Search Method to solve Economic Dispa...   
3  A Direct Search Method to solve Economic Dispa...   
4  A Direct Search Method to solve Economic Dispa...   

                                            positive  
0  An evolutionary recurrent network which automa...  
1  Dynamic economic dispatch (DED) is one of the ...  
2  It's not surprisingly when entering this site ...  
3  In this paper, we introduce a new parameter, c...  
4  This paper proposes a recurrent fuzzy neural n...  


In [ ]:
val_positive = val_qrels[
    val_qrels["score"] == 1
][["query_id", "doc_id"]].copy()

val_pairs = val_positive.copy()

val_pairs["query"] = val_pairs["query_id"].map(
    query_lookup
)

val_pairs["positive"] = val_pairs["doc_id"].map(
    doc_lookup
)

val_pairs = val_pairs[
    ["query", "positive"]
].reset_index(drop=True)

print("Validation pairs:", val_pairs.shape)
print(val_pairs.head())

Validation pairs: (495, 2)
                                 query  \
0  Social engineering attack framework   
1  Social engineering attack framework   
2  Social engineering attack framework   
3  Social engineering attack framework   
4  Social engineering attack framework   

                                            positive  
0  1 Why develop an ontology? In recent years the...  
1  Social engineering is a type of attack that al...  
2                                                     
3  Trusted people can fail to be trustworthy when...  
4  This paper is intended to serve as a comprehen...  


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

save_dir = "/content/drive/MyDrive/Start_LLM/Enterprise_Semantic_Search_Engine/Data"
os.makedirs(save_dir, exist_ok=True)

train_pairs.to_parquet(
    f"{save_dir}/train_pairs.parquet",
    index=False
)

val_pairs.to_parquet(
    f"{save_dir}/val_pairs.parquet",
    index=False
)

train_hard_negatives.to_parquet(
    f"{save_dir}/hard_negatives.parquet",
    index=False
)

test_qrels.to_parquet(
    f"{save_dir}/test_qrels.parquet",
    index=False
)


print("Datasets saved successfully.")

Datasets saved successfully.


In [ ]:
test_qrels.to_parquet(
    f"{save_dir}/test_qrels.parquet",
    index=False
)

print("Test qrels saved.")

Test qrels saved.


In [ ]:
import math

ndcg_scores = []

for query_idx, labels in enumerate(retrieval_labels):

    query_id = queries_df.iloc[query_idx]["_id"]

    # Number of truly relevant documents for this query
    total_relevant = len(qrels_dict[query_id])

    # DCG@10
    dcg = 0.0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            dcg += 1 / math.log2(rank + 1)

    # IDCG@10
    ideal_relevant = min(total_relevant, len(labels))

    idcg = 0.0

    for rank in range(1, ideal_relevant + 1):
        idcg += 1 / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0.0

    ndcg_scores.append(ndcg)

mean_ndcg_at_10 = sum(ndcg_scores) / len(ndcg_scores)

print(f"Standard nDCG@10: {mean_ndcg_at_10:.4f}")

Standard nDCG@10: 0.2040


In [ ]:
print("=== Pre-trained Baseline ===")
print(f"Recall@10 : {mean_recall_at_10:.4f}")
print(f"MRR@10    : {mean_mrr_at_10:.4f}")
print(f"nDCG@10   : {mean_ndcg_at_10:.4f}")

=== Pre-trained Baseline ===
Recall@10 : 0.2207
MRR@10    : 0.3431
nDCG@10   : 0.2040


Train

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_pairs[["query", "positive"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_pairs[["query", "positive"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['query', 'positive'],
    num_rows: 3944
})
Dataset({
    features: ['query', 'positive'],
    num_rows: 495
})


In [ ]:
from sentence_transformers import losses

loss = losses.MultipleNegativesRankingLoss(model)

/tmp/ipykernel_2272/2962838251.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses


In [ ]:
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers import losses

train_dataset = Dataset.from_pandas(
    train_pairs[["query", "positive"]],
    preserve_index=False
)

loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="/tmp/minilm_finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=50,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

/tmp/ipykernel_2272/4202127383.py:2: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [ ]:
trainer.train()

Step,Training Loss
50,1.460146
100,1.238806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=124, training_loss=1.330938200796804, metrics={'train_runtime': 17.2628, 'train_samples_per_second': 228.468, 'train_steps_per_second': 7.183, 'total_flos': 0.0, 'train_loss': 1.330938200796804, 'epoch': 1.0})

In [ ]:
fine_tuned_corpus_embeddings = model.encode(
    corpus_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

fine_tuned_query_embeddings = model.encode(
    queries_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Corpus:", fine_tuned_corpus_embeddings.shape)
print("Queries:", fine_tuned_query_embeddings.shape)

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus: torch.Size([25657, 384])
Queries: torch.Size([1000, 384])


In [ ]:
fine_tuned_hits = util.semantic_search(
    fine_tuned_query_embeddings,
    fine_tuned_corpus_embeddings,
    top_k=10
)

print("Queries:", len(fine_tuned_hits))
print("Top-k per query:", len(fine_tuned_hits[0]))

Queries: 1000
Top-k per query: 10


In [ ]:
fine_tuned_retrieval_labels = []

for query_idx, hits in enumerate(fine_tuned_hits):

    query_id = queries_df.iloc[query_idx]["_id"]
    relevant_docs = qrels_dict[query_id]

    labels = []

    for hit in hits:
        doc_id = corpus_id_to_doc_id[hit["corpus_id"]]

        if doc_id in relevant_docs:
            labels.append(1)
        else:
            labels.append(0)

    fine_tuned_retrieval_labels.append(labels)

print("Queries:", len(fine_tuned_retrieval_labels))
print("Labels for first query:", fine_tuned_retrieval_labels[0])

Queries: 1000
Labels for first query: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
import math

# Recall@10
recall_scores = []

# MRR@10
mrr_scores = []

# nDCG@10
ndcg_scores = []


for query_idx, labels in enumerate(fine_tuned_retrieval_labels):

    query_id = queries_df.iloc[query_idx]["_id"]
    total_relevant = len(qrels_dict[query_id])

    # -------------------------
    # Recall@10
    # -------------------------
    retrieved_relevant = sum(labels)

    recall = retrieved_relevant / total_relevant
    recall_scores.append(recall)

    # -------------------------
    # MRR@10
    # -------------------------
    reciprocal_rank = 0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            reciprocal_rank = 1 / rank
            break

    mrr_scores.append(reciprocal_rank)

    # -------------------------
    # nDCG@10
    # -------------------------
    dcg = 0.0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            dcg += 1 / math.log2(rank + 1)

    ideal_relevant = min(total_relevant, len(labels))

    idcg = 0.0

    for rank in range(1, ideal_relevant + 1):
        idcg += 1 / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0.0
    ndcg_scores.append(ndcg)


fine_tuned_recall = sum(recall_scores) / len(recall_scores)
fine_tuned_mrr = sum(mrr_scores) / len(mrr_scores)
fine_tuned_ndcg = sum(ndcg_scores) / len(ndcg_scores)


print(f"Fine-tuned Recall@10 : {fine_tuned_recall:.4f}")
print(f"Fine-tuned MRR@10    : {fine_tuned_mrr:.4f}")
print(f"Fine-tuned nDCG@10   : {fine_tuned_ndcg:.4f}")

Fine-tuned Recall@10 : 0.2288
Fine-tuned MRR@10    : 0.3474
Fine-tuned nDCG@10   : 0.2103


In [7]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
import pandas as pd

save_dir =  "/content/drive/MyDrive/Start_LLM/Enterprise_Semantic_Search_Engine/Data"

train_pairs = pd.read_parquet(f"{save_dir}/train_pairs.parquet")
val_pairs = pd.read_parquet(f"{save_dir}/val_pairs.parquet")
test_qrels = pd.read_parquet(f"{save_dir}/test_qrels.parquet")
hard_negatives = pd.read_parquet(f"{save_dir}/hard_negatives.parquet")

print("train_pairs:", train_pairs.shape)
print("val_pairs:", val_pairs.shape)
print("test_qrels:", test_qrels.shape)
print("hard_negatives:", hard_negatives.shape)

train_pairs: (3944, 2)
val_pairs: (495, 2)
test_qrels: (2989, 3)
hard_negatives: (38449, 3)


In [ ]:
hard_negatives = hard_negatives.sort_values(
    ["query_id", "score"],
    ascending=[True, False]
)

hard_negative_pool = (
    hard_negatives
    .groupby("query_id")
    .head(5)
    .reset_index(drop=True)
)

print("Hard-negative pool:", hard_negative_pool.shape)
print(hard_negative_pool.head())

Hard-negative pool: (4000, 3)
                                   query_id  \
0  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
1  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
2  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
3  01273bd34dacfe9ef887b320f36934d2f9fa9b34   
4  01273bd34dacfe9ef887b320f36934d2f9fa9b34   

                                     doc_id     score  
0  a0fd813b9218813e1b020d03a3099de7677dd145  0.407742  
1  802b80852996d87dc16082b86f6e77115eb6c9a6  0.385802  
2  0faeda106e40a43a208f76824d8001baac2eefbc  0.370300  
3  d1b86fa7c4160811de7529bff6bb1cfc206ac1d0  0.368244  
4  f7686113611ac3d03c1cd412ebf46ba5e35b3071  0.361985  


In [ ]:
# Lookup dictionaries
query_lookup = queries_df.set_index("_id")["text"].to_dict()
doc_lookup = corpus_df.set_index("_id")["text"].to_dict()

triplets = []

for query_id, positives in train_pairs.groupby(
    train_pairs["query"].map(
        {v: k for k, v in query_lookup.items()}
    )
):
    pass

In [ ]:
# Map text -> ID
query_text_to_id = queries_df.set_index("text")["_id"].to_dict()
doc_text_to_id = corpus_df.set_index("text")["_id"].to_dict()

# Add IDs back to train_pairs
train_pairs["query_id"] = train_pairs["query"].map(query_text_to_id)
train_pairs["doc_id"] = train_pairs["positive"].map(doc_text_to_id)

# Build triplets
triplets = []

for query_id, positives in train_pairs.groupby("query_id"):

    negatives = hard_negative_pool[
        hard_negative_pool["query_id"] == query_id
    ].copy()

    n = min(len(positives), len(negatives))

    negatives = negatives.sample(
        n=n,
        random_state=42
    ).reset_index(drop=True)

    positives = positives.reset_index(drop=True)

    for i in range(n):
        triplets.append({
            "query": positives.iloc[i]["query"],
            "positive": positives.iloc[i]["positive"],
            "negative": doc_lookup[negatives.iloc[i]["doc_id"]]
        })

train_triplets = pd.DataFrame(triplets)

print("Training triplets:", train_triplets.shape)
print(train_triplets.head(3))

Training triplets: (3944, 3)
                                         query  \
0  Image-Guided Nanopositioning Scheme for SEM   
1  Image-Guided Nanopositioning Scheme for SEM   
2  Image-Guided Nanopositioning Scheme for SEM   

                                            positive  \
0  Robotics continues to provide researchers with...   
1  In this paper, we have derived analytic expres...   
2  Optical flow cannot be computed locally, since...   

                                            negative  
0  In this article, a methodology to extract Flas...  
1  Multiple problems, including high computationa...  
2  We demonstrate that metal-insulator-metal conf...  


In [ ]:
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from datasets import Dataset

# Start from the original pretrained model
model_hn = SentenceTransformer("all-MiniLM-L6-v2")

# Convert triplets to Hugging Face Dataset
train_triplet_dataset = Dataset.from_pandas(
    train_triplets[["query", "positive", "negative"]],
    preserve_index=False
)

# Triplet loss
loss = losses.TripletLoss(model=model_hn)

print(train_triplet_dataset)

/tmp/ipykernel_1803/729402478.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, losses
/tmp/ipykernel_1803/729402478.py:3: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dataset({
    features: ['query', 'positive', 'negative'],
    num_rows: 3944
})


In [ ]:
args_hn = SentenceTransformerTrainingArguments(
    output_dir="/tmp/minilm_hard_negative",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    warmup_ratio=0.1,
    fp16=True,
    logging_strategy="epoch",
)

trainer_hn = SentenceTransformerTrainer(
    model=model_hn,
    args=args_hn,
    train_dataset=train_triplet_dataset,
    loss=loss,
)

trainer_hn.train()

Step,Training Loss
124,4.644416
248,4.685853
372,4.539508
496,4.481961
620,4.435307


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=620, training_loss=4.5574089788621475, metrics={'train_runtime': 153.8097, 'train_samples_per_second': 128.21, 'train_steps_per_second': 4.031, 'total_flos': 0.0, 'train_loss': 4.5574089788621475, 'epoch': 5.0})

In [ ]:
fine_tuned_hn_corpus_embeddings = model_hn.encode(
    corpus_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

fine_tuned_hn_query_embeddings = model_hn.encode(
    queries_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Corpus:", fine_tuned_hn_corpus_embeddings.shape)
print("Queries:", fine_tuned_hn_query_embeddings.shape)

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus: torch.Size([25657, 384])
Queries: torch.Size([1000, 384])


In [ ]:
from sentence_transformers import util

fine_tuned_hits = util.semantic_search(
    fine_tuned_hn_query_embeddings,
    fine_tuned_hn_corpus_embeddings,
    top_k=10
)

print("Queries:", len(fine_tuned_hits))
print("Top-k per query:", len(fine_tuned_hits[0]))

Queries: 1000
Top-k per query: 10


In [ ]:
fine_tuned_retrieval_labels = []

for query_idx, hits in enumerate(fine_tuned_hits):

    query_id = queries_df.iloc[query_idx]["_id"]
    relevant_docs = qrels_dict[query_id]

    labels = []

    for hit in hits:
        doc_id = corpus_id_to_doc_id[hit["corpus_id"]]

        if doc_id in relevant_docs:
            labels.append(1)
        else:
            labels.append(0)

    fine_tuned_retrieval_labels.append(labels)

print("Queries:", len(fine_tuned_retrieval_labels))
print("Labels for first query:", fine_tuned_retrieval_labels[0])

Queries: 1000
Labels for first query: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
import math

# Recall@10
recall_scores = []

# MRR@10
mrr_scores = []

# nDCG@10
ndcg_scores = []


for query_idx, labels in enumerate(fine_tuned_retrieval_labels):

    query_id = queries_df.iloc[query_idx]["_id"]
    total_relevant = len(qrels_dict[query_id])

    # -------------------------
    # Recall@10
    # -------------------------
    retrieved_relevant = sum(labels)

    recall = retrieved_relevant / total_relevant
    recall_scores.append(recall)

    # -------------------------
    # MRR@10
    # -------------------------
    reciprocal_rank = 0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            reciprocal_rank = 1 / rank
            break

    mrr_scores.append(reciprocal_rank)

    # -------------------------
    # nDCG@10
    # -------------------------
    dcg = 0.0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            dcg += 1 / math.log2(rank + 1)

    ideal_relevant = min(total_relevant, len(labels))

    idcg = 0.0

    for rank in range(1, ideal_relevant + 1):
        idcg += 1 / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0.0
    ndcg_scores.append(ndcg)


fine_tuned_recall = sum(recall_scores) / len(recall_scores)
fine_tuned_mrr = sum(mrr_scores) / len(mrr_scores)
fine_tuned_ndcg = sum(ndcg_scores) / len(ndcg_scores)


print(f"Fine-tuned Recall@10 : {fine_tuned_recall:.4f}")
print(f"Fine-tuned MRR@10    : {fine_tuned_mrr:.4f}")
print(f"Fine-tuned nDCG@10   : {fine_tuned_ndcg:.4f}")

Fine-tuned Recall@10 : 0.0014
Fine-tuned MRR@10    : 0.0119
Fine-tuned nDCG@10   : 0.0041


In [ ]:
query_idx = 0

print("Query:")
print(queries_df.iloc[query_idx]["text"])

print("\nTop 5 results:")

for hit in fine_tuned_hits[query_idx][:5]:
    doc_idx = hit["corpus_id"]

    print(
        f"\nScore: {hit['score']:.4f}"
        f"\nDoc: {corpus_df.iloc[doc_idx]['text'][:300]}"
    )

Query:
A Direct Search Method to solve Economic Dispatch Problem with Valve-Point Effect

Top 5 results:

Score: 0.9998
Doc: ÐA new view-based approach to the representation and recognition of human movement is presented. The basis of the representation is a temporal templateÐa static vector-image where the vector value at each point is a function of the motion properties at the corresponding spatial location in an image 

Score: 0.9998
Doc: The Byzantine Generals Problem requires processes to reach agreement upon a value even though some of them may fad. It is weakened by allowing them to agree upon an "incorrect" value if a failure occurs. The transaction eormmt problem for a distributed database Js a special case of the weaker proble

Score: 0.9998
Doc: In standard fractal terrain models based on fractional Brownian motion the statistical character of the surface is, by design, the same everywhere. A new approach to the synthesis of fractal terrain height fields is presented which, 

In [ ]:
import torch
from sentence_transformers import util

# Encode a few unrelated documents
sample_docs = corpus_df["text"].iloc[:10].tolist()

sample_embeddings = model_hn.encode(
    sample_docs,
    convert_to_tensor=True,
    normalize_embeddings=True
)

similarity_matrix = util.cos_sim(
    sample_embeddings,
    sample_embeddings
)

print("Similarity matrix:")
print(similarity_matrix[:5, :5])

Similarity matrix:
tensor([[ 1.0000,  0.9995, -0.9976,  0.7516,  0.9996],
        [ 0.9995,  1.0000, -0.9975,  0.7503,  0.9997],
        [-0.9976, -0.9975,  1.0000, -0.7656, -0.9980],
        [ 0.7516,  0.7503, -0.7656,  1.0000,  0.7522],
        [ 0.9996,  0.9997, -0.9980,  0.7522,  1.0000]], device='cuda:0')


In [ ]:
import sentence_transformers

print(sentence_transformers.__version__)

5.7.0


In [ ]:
import inspect
from sentence_transformers import losses

print(inspect.signature(losses.MultipleNegativesRankingLoss))
print(inspect.signature(losses.CachedMultipleNegativesRankingLoss))

(model: 'SentenceTransformer', scale: 'float' = 20.0, similarity_fct: 'Callable[[Tensor, Tensor], Tensor]' = <function cos_sim at 0x7bcaeec368e0>, gather_across_devices: 'bool' = False, directions: "tuple[Literal['query_to_doc', 'query_to_query', 'doc_to_query', 'doc_to_doc'], ...]" = ('query_to_doc',), partition_mode: "Literal['joint', 'per_direction']" = 'joint', hardness_mode: "Literal['in_batch_negatives', 'hard_negatives', 'all_negatives'] | None" = None, hardness_strength: 'float' = 0.0) -> 'None'
(model: 'SentenceTransformer', scale: 'float' = 20.0, similarity_fct: 'Callable[[Tensor, Tensor], Tensor]' = <function cos_sim at 0x7bcaeec368e0>, mini_batch_size: 'int' = 32, mini_batch_num_tokens: 'int | None' = None, gather_across_devices: 'bool' = False, directions: "tuple[Literal['query_to_doc', 'query_to_query', 'doc_to_query', 'doc_to_doc'], ...]" = ('query_to_doc',), partition_mode: "Literal['joint', 'per_direction']" = 'joint', show_progress_bar: 'bool' = False, hardness_mode: 

In [ ]:
from sentence_transformers import SentenceTransformer, losses

model_hn_mnrl = SentenceTransformer("all-MiniLM-L6-v2")

loss_hn = losses.MultipleNegativesRankingLoss(
    model=model_hn_mnrl,
    hardness_mode="hard_negatives",
    hardness_strength=1.0
)

print(loss_hn)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MultipleNegativesRankingLoss(
  (model): SentenceTransformer(
    (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
    (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
    (2): Normalize({})
  )
)


In [ ]:
print(train_triplet_dataset.features)

{'query': Value('string'), 'positive': Value('string'), 'negative': Value('string')}


In [ ]:
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

args_hn = SentenceTransformerTrainingArguments(
    output_dir="/tmp/minilm_hard_negative_mnrl",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=50,
)

trainer_hn = SentenceTransformerTrainer(
    model=model_hn_mnrl,
    args=args_hn,
    train_dataset=train_triplet_dataset,
    loss=loss_hn,
)

trainer_hn.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
50,3.316516
100,2.682433


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=124, training_loss=2.9121935752130326, metrics={'train_runtime': 31.3682, 'train_samples_per_second': 125.732, 'train_steps_per_second': 3.953, 'total_flos': 0.0, 'train_loss': 2.9121935752130326, 'epoch': 1.0})

In [ ]:
fine_tuned_hn_corpus_embeddings = model_hn.encode(
    corpus_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

fine_tuned_hn_query_embeddings = model_hn.encode(
    queries_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Corpus:", fine_tuned_hn_corpus_embeddings.shape)
print("Queries:", fine_tuned_hn_query_embeddings.shape)

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus: torch.Size([25657, 384])
Queries: torch.Size([1000, 384])


In [ ]:
from sentence_transformers import util

fine_tuned_hits = util.semantic_search(
    fine_tuned_hn_query_embeddings,
    fine_tuned_hn_corpus_embeddings,
    top_k=10
)

print("Queries:", len(fine_tuned_hits))
print("Top-k per query:", len(fine_tuned_hits[0]))


fine_tuned_retrieval_labels = []

for query_idx, hits in enumerate(fine_tuned_hits):

    query_id = queries_df.iloc[query_idx]["_id"]
    relevant_docs = qrels_dict[query_id]

    labels = []

    for hit in hits:
        doc_id = corpus_id_to_doc_id[hit["corpus_id"]]

        if doc_id in relevant_docs:
            labels.append(1)
        else:
            labels.append(0)

    fine_tuned_retrieval_labels.append(labels)

print("Queries:", len(fine_tuned_retrieval_labels))
print("Labels for first query:", fine_tuned_retrieval_labels[0])


Queries: 1000
Top-k per query: 10
Queries: 1000
Labels for first query: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
import math

# Recall@10
recall_scores = []

# MRR@10
mrr_scores = []

# nDCG@10
ndcg_scores = []


for query_idx, labels in enumerate(fine_tuned_retrieval_labels):

    query_id = queries_df.iloc[query_idx]["_id"]
    total_relevant = len(qrels_dict[query_id])

    # -------------------------
    # Recall@10
    # -------------------------
    retrieved_relevant = sum(labels)

    recall = retrieved_relevant / total_relevant
    recall_scores.append(recall)

    # -------------------------
    # MRR@10
    # -------------------------
    reciprocal_rank = 0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            reciprocal_rank = 1 / rank
            break

    mrr_scores.append(reciprocal_rank)

    # -------------------------
    # nDCG@10
    # -------------------------
    dcg = 0.0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            dcg += 1 / math.log2(rank + 1)

    ideal_relevant = min(total_relevant, len(labels))

    idcg = 0.0

    for rank in range(1, ideal_relevant + 1):
        idcg += 1 / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0.0
    ndcg_scores.append(ndcg)


fine_tuned_recall = sum(recall_scores) / len(recall_scores)
fine_tuned_mrr = sum(mrr_scores) / len(mrr_scores)
fine_tuned_ndcg = sum(ndcg_scores) / len(ndcg_scores)


print(f"Fine-tuned Recall@10 : {fine_tuned_recall:.4f}")
print(f"Fine-tuned MRR@10    : {fine_tuned_mrr:.4f}")
print(f"Fine-tuned nDCG@10   : {fine_tuned_ndcg:.4f}")

Fine-tuned Recall@10 : 0.0014
Fine-tuned MRR@10    : 0.0119
Fine-tuned nDCG@10   : 0.0041


In [ ]:
print("New model:", model_hn_mnrl)
print("Old Triplet model:", model_hn)

print(
    "New model and old model are same object:",
    model_hn_mnrl is model_hn
)

New model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
Old Triplet model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
New model and old model are same object: False


In [ ]:
sample_docs = corpus_df["text"].iloc[:10].tolist()

sample_embeddings_new = model_hn_mnrl.encode(
    sample_docs,
    convert_to_tensor=True,
    normalize_embeddings=True
)

similarity_matrix_new = util.cos_sim(
    sample_embeddings_new,
    sample_embeddings_new
)

print(similarity_matrix_new[:5, :5])

tensor([[1.0000, 0.7496, 0.8012, 0.7971, 0.8667],
        [0.7496, 1.0000, 0.6397, 0.7317, 0.6735],
        [0.8012, 0.6397, 1.0000, 0.7445, 0.7046],
        [0.7971, 0.7317, 0.7445, 1.0000, 0.6720],
        [0.8667, 0.6735, 0.7046, 0.6720, 1.0000]], device='cuda:0')


all-mpnet-base-v2

In [ ]:
from sentence_transformers import SentenceTransformer

mpnet_model = SentenceTransformer("all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
mpnet_corpus_embeddings = mpnet_model.encode(
    corpus_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

mpnet_query_embeddings = mpnet_model.encode(
    queries_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Corpus:", mpnet_corpus_embeddings.shape)
print("Queries:", mpnet_query_embeddings.shape)

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus: torch.Size([25657, 768])
Queries: torch.Size([1000, 768])


In [ ]:
from sentence_transformers import util

mpnet_hits = util.semantic_search(
    mpnet_query_embeddings,
    mpnet_corpus_embeddings,
    top_k=10
)

In [ ]:

print("Queries:", len(mpnet_hits))
print("Top-k per query:", len(mpnet_hits[0]))


fine_tuned_retrieval_labels = []

for query_idx, hits in enumerate(mpnet_hits):

    query_id = queries_df.iloc[query_idx]["_id"]
    relevant_docs = qrels_dict[query_id]

    labels = []

    for hit in hits:
        doc_id = corpus_id_to_doc_id[hit["corpus_id"]]

        if doc_id in relevant_docs:
            labels.append(1)
        else:
            labels.append(0)

    fine_tuned_retrieval_labels.append(labels)

print("Queries:", len(fine_tuned_retrieval_labels))
print("Labels for first query:", fine_tuned_retrieval_labels[0])


Queries: 1000
Top-k per query: 10
Queries: 1000
Labels for first query: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
import math

# Recall@10
recall_scores = []

# MRR@10
mrr_scores = []

# nDCG@10
ndcg_scores = []


for query_idx, labels in enumerate(fine_tuned_retrieval_labels):

    query_id = queries_df.iloc[query_idx]["_id"]
    total_relevant = len(qrels_dict[query_id])

    # -------------------------
    # Recall@10
    # -------------------------
    retrieved_relevant = sum(labels)

    recall = retrieved_relevant / total_relevant
    recall_scores.append(recall)

    # -------------------------
    # MRR@10
    # -------------------------
    reciprocal_rank = 0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            reciprocal_rank = 1 / rank
            break

    mrr_scores.append(reciprocal_rank)

    # -------------------------
    # nDCG@10
    # -------------------------
    dcg = 0.0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            dcg += 1 / math.log2(rank + 1)

    ideal_relevant = min(total_relevant, len(labels))

    idcg = 0.0

    for rank in range(1, ideal_relevant + 1):
        idcg += 1 / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0.0
    ndcg_scores.append(ndcg)


fine_tuned_recall = sum(recall_scores) / len(recall_scores)
fine_tuned_mrr = sum(mrr_scores) / len(mrr_scores)
fine_tuned_ndcg = sum(ndcg_scores) / len(ndcg_scores)


print(f"Fine-tuned Recall@10 : {fine_tuned_recall:.4f}")
print(f"Fine-tuned MRR@10    : {fine_tuned_mrr:.4f}")
print(f"Fine-tuned nDCG@10   : {fine_tuned_ndcg:.4f}")

Fine-tuned Recall@10 : 0.0402
Fine-tuned MRR@10    : 0.3612
Fine-tuned nDCG@10   : 0.1440


BAAI/bge-base-en-v1.5

In [ ]:
from sentence_transformers import SentenceTransformer

bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
bge_corpus_embeddings = bge_model.encode(
    corpus_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

bge_query_embeddings = bge_model.encode(
    queries_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Corpus:", bge_corpus_embeddings.shape)
print("Queries:", bge_query_embeddings.shape)

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus: torch.Size([25657, 768])
Queries: torch.Size([1000, 768])


In [ ]:
from sentence_transformers import util

bge_hits = util.semantic_search(
    bge_query_embeddings,
    bge_corpus_embeddings,
    top_k=10
)

print("Queries:", len(bge_hits))
print("Top-k:", len(bge_hits[0]))

Queries: 1000
Top-k: 10


In [ ]:
bge_retrieval_labels = []

for query_idx, hits in enumerate(bge_hits):

    query_id = queries_df.iloc[query_idx]["_id"]
    relevant_docs = qrels_dict[query_id]

    labels = []

    for hit in hits:

        doc_id = corpus_id_to_doc_id[hit["corpus_id"]]

        if doc_id in relevant_docs:
            labels.append(1)
        else:
            labels.append(0)

    bge_retrieval_labels.append(labels)

In [ ]:
# Recall@10
recall_scores = []

for query_idx, labels in enumerate(bge_retrieval_labels):

    query_id = queries_df.iloc[query_idx]["_id"]
    total_relevant = len(qrels_dict[query_id])
    retrieved_relevant = sum(labels)

    recall = retrieved_relevant / total_relevant
    recall_scores.append(recall)

bge_recall = sum(recall_scores) / len(recall_scores)


# MRR@10
mrr_scores = []

for labels in bge_retrieval_labels:

    reciprocal_rank = 0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            reciprocal_rank = 1 / rank
            break

    mrr_scores.append(reciprocal_rank)

bge_mrr = sum(mrr_scores) / len(mrr_scores)


# nDCG@10
import math

ndcg_scores = []

for query_idx, labels in enumerate(bge_retrieval_labels):

    query_id = queries_df.iloc[query_idx]["_id"]
    total_relevant = len(qrels_dict[query_id])

    dcg = 0.0

    for rank, label in enumerate(labels, start=1):
        if label == 1:
            dcg += 1 / math.log2(rank + 1)

    ideal_relevant = min(total_relevant, len(labels))

    idcg = 0.0

    for rank in range(1, ideal_relevant + 1):
        idcg += 1 / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0.0

    ndcg_scores.append(ndcg)

bge_ndcg = sum(ndcg_scores) / len(ndcg_scores)


print(f"BGE Recall@10 : {bge_recall:.4f}")
print(f"BGE MRR@10    : {bge_mrr:.4f}")
print(f"BGE nDCG@10   : {bge_ndcg:.4f}")

BGE Recall@10 : 0.0384
BGE MRR@10    : 0.3494
BGE nDCG@10   : 0.1386


"Qwen/Qwen3-Embedding-0.6B"

In [ ]:
import gc
import torch

# حذف مدل‌ها
for name in [
    "model",
    "mpnet_model",
    "model_bge",
    "bge_model",
    "model_hn",
    "model_hn_mnrl",
]:
    globals().pop(name, None)

# حذف embeddingهای قبلی
for name in [
    "corpus_embeddings",
    "query_embeddings",
    "mpnet_corpus_embeddings",
    "mpnet_query_embeddings",
    "bge_corpus_embeddings",
    "bge_query_embeddings",
]:
    globals().pop(name, None)

# حذف retrieval results
for name in [
    "all_hits",
    "mpnet_hits",
    "bge_hits",
    "hard_negative_hits",
]:
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

3.1709041595458984 GB allocated
3.77734375 GB reserved


In [ ]:
from sentence_transformers import SentenceTransformer

qwen_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B"
)

print(qwen_model)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Qwen3Model'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'lasttoken', 'include_prompt': True})
  (2): Normalize({})
)


In [ ]:
import gc
import torch

# حذف مدل‌های قبلی
del qwen_model

# اگر مدل‌های embedding قبلی هنوز در حافظه هستند، آن‌ها را هم حذف کن
# del model
# del mpnet_model
# del model_bge

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU allocated: 3.17 GB
GPU reserved:  4.20 GB


In [ ]:
import torch
from sentence_transformers import SentenceTransformer

qwen_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"torch_dtype": torch.float16}
)

print(qwen_model)
print(
    f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Qwen3Model'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'lasttoken', 'include_prompt': True})
  (2): Normalize({})
)
Allocated: 4.28 GB


In [ ]:
test_embedding = qwen_model.encode(
    corpus_df["text"].iloc[:10].tolist(),
    batch_size=2,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print(test_embedding.shape)

torch.Size([10, 1024])


In [ ]:
qwen_corpus_embeddings = qwen_model.encode(
    corpus_df["text"].tolist(),
    batch_size=16,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Corpus:", qwen_corpus_embeddings.shape)

Batches:   0%|          | 0/1604 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.45 GiB. GPU 0 has a total capacity of 14.56 GiB of which 213.81 MiB is free. Including non-PyTorch memory, this process has 14.35 GiB memory in use. Of the allocated memory 12.27 GiB is allocated by PyTorch, and 1.95 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
import gc
import torch

# مدل‌های embedding قبلی
for name in [
    "qwen_model",
    "model",
    "mpnet_model",
    "model_bge",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU allocated: 8.64 GB
GPU reserved:  11.90 GB


In [ ]:
from sentence_transformers import SentenceTransformer

nomic_model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v2-moe",
    trust_remote_code=True
)

print(nomic_model)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/554 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.10k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.48k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'NomicBertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [ ]:
test_docs = [
    "search_document: " + text
    for text in corpus_df["text"].iloc[:10]
]

test_queries = [
    "search_query: " + text
    for text in queries_df["text"].iloc[:3]
]

doc_emb = nomic_model.encode(
    test_docs,
    batch_size=2,
    normalize_embeddings=True
)

query_emb = nomic_model.encode(
    test_queries,
    batch_size=2,
    normalize_embeddings=True
)

print("Document embeddings:", doc_emb.shape)
print("Query embeddings:", query_emb.shape)

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Document embeddings: (10, 768)
Query embeddings: (3, 768)


chunking

In [9]:
from transformers import AutoTokenizer
import pandas as pd


tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_size = 128
overlap = 32
stride = chunk_size - overlap

chunks = []

for _, row in corpus_df.iterrows():
    doc_id = row["_id"]
    text = row["text"]

    # Tokenize without truncation
    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    # Create overlapping chunks
    for start in range(0, len(tokens), stride):
        chunk_tokens = tokens[start:start + chunk_size]

        if not chunk_tokens:
            break

        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append({
            "doc_id": doc_id,
            "chunk_id": start // stride,
            "text": chunk_text
        })

        # Stop when we've reached the end
        if start + chunk_size >= len(tokens):
            break

chunks_df = pd.DataFrame(chunks)

print("Original documents:", len(corpus_df))
print("Total chunks:", len(chunks_df))
print()
print(chunks_df.head())

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (621 > 512). Running this sequence through the model will result in indexing errors


Original documents: 25657
Total chunks: 63284

                                     doc_id  chunk_id  \
0  632589828c8b9fca2c3a59e97451fde8fa7d188d         0   
1  632589828c8b9fca2c3a59e97451fde8fa7d188d         1   
2  632589828c8b9fca2c3a59e97451fde8fa7d188d         2   
3  86e87db2dab958f1bd5877dc7d5b8105d6e31e46         0   
4  86e87db2dab958f1bd5877dc7d5b8105d6e31e46         1   

                                                text  
0  an evolutionary recurrent network which automa...  
1  of elite strategy is adopted in hgapso, where ...  
2  elites constitute half of the population in th...  
3  dynamic economic dispatch ( ded ) is one of th...  
4  , which can give a good direction to the optim...  


In [10]:
print("\nChunks per document:")
print(chunks_df.groupby("doc_id").size().describe())

print("\nUnique documents:", chunks_df["doc_id"].nunique())
print("Unique chunks:", len(chunks_df))

print("\nExample document:")
example_doc = chunks_df["doc_id"].iloc[0]
print(chunks_df[chunks_df["doc_id"] == example_doc].to_string(index=False))


Chunks per document:
count    25313.000000
mean         2.500059
std          1.760492
min          1.000000
25%          2.000000
50%          2.000000
75%          3.000000
max         59.000000
dtype: float64

Unique documents: 25313
Unique chunks: 63284

Example document:
                                  doc_id  chunk_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 text
632589828c

In [11]:
# Find documents that produced no chunks
chunked_doc_ids = set(chunks_df["doc_id"])

missing_docs = corpus_df[
    ~corpus_df["_id"].isin(chunked_doc_ids)
]

print("Missing documents:", len(missing_docs))
print()
print(missing_docs[["_id", "title", "text"]].head(20).to_string(index=False))

Missing documents: 344

                                     _id                                                                                                                                                       title text
e275f643c97ca1f4c7715635bb72cf02df928d06                                                                                                                                  From Databases to Big Data     
1e55bb7c095d3ea15bccb3df920c546ec54c86b5                                                                                                Enterprise resource planning: A taxonomy of critical factors     
57bbbfea63019a57ef658a27622c357978400a50                                                                                             U-Net: Convolutional Networks for Biomedical Image Segmentation     
bf003bb2d52304fea114d824bc0bf7bfbc7c3106                                                                                                                               D

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
     "sentence-transformers/all-MiniLM-L6-v2"
)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
# Encode all chunks with the fine-tuned MiniLM model

chunk_texts = chunks_df["text"].tolist()

chunk_embeddings = model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Chunks:", len(chunks_df))
print("Embeddings:", chunk_embeddings.shape)

Batches:   0%|          | 0/1978 [00:00<?, ?it/s]

Chunks: 63284
Embeddings: (63284, 384)


In [16]:
# Encode queries with the same fine-tuned MiniLM model

query_texts = queries_df["text"].tolist()

query_embeddings = model.encode(
    query_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Queries:", len(queries_df))
print("Query embeddings:", query_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Queries: 1000
Query embeddings: (1000, 384)


In [20]:
import numpy as np

top_k = 10

# Chunk-level similarity
similarities = query_embeddings @ chunk_embeddings.T

# Retrieve more chunks than k so we can collect 10 unique documents
candidate_k = 200
candidate_indices = np.argsort(-similarities, axis=1)[:, :candidate_k]

recall_scores = []
mrr_scores = []
ndcg_scores = []

for query_idx, query_id in enumerate(queries_df["_id"]):

    relevant_docs = qrels_dict.get(query_id, set())

    # Convert retrieved chunks -> unique original documents
    retrieved_docs = []

    for chunk_idx in candidate_indices[query_idx]:
        doc_id = chunks_df.iloc[chunk_idx]["doc_id"]

        if doc_id not in retrieved_docs:
            retrieved_docs.append(doc_id)

        if len(retrieved_docs) == top_k:
            break

    # Binary relevance
    relevance = [
        1 if doc_id in relevant_docs else 0
        for doc_id in retrieved_docs
    ]

    # Recall@10
    num_relevant_retrieved = sum(relevance)
    recall = num_relevant_retrieved / len(relevant_docs)
    recall_scores.append(min(recall, 1.0))

    # MRR@10
    rr = 0.0

    for rank, rel in enumerate(relevance, start=1):
        if rel == 1:
            rr = 1.0 / rank
            break

    mrr_scores.append(rr)

    # nDCG@10
    dcg = sum(
        rel / np.log2(rank + 1)
        for rank, rel in enumerate(relevance, start=1)
    )

    ideal_relevance = [1] * min(len(relevant_docs), top_k)

    idcg = sum(
        rel / np.log2(rank + 1)
        for rank, rel in enumerate(ideal_relevance, start=1)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0
    ndcg_scores.append(ndcg)


print(f"Recall@10: {np.mean(recall_scores):.4f}")
print(f"MRR@10:    {np.mean(mrr_scores):.4f}")
print(f"nDCG@10:   {np.mean(ndcg_scores):.4f}")

Recall@10: 0.0345
MRR@10:    0.3320
nDCG@10:   0.1260
